# Introduction

In this assignment, you are asked to solve several cases, and each of them must be solved comprehensively. As the name suggests, in this assignment you need to apply parametric or non-parametric inference methods. It means, before deciding to use any method, you must check if the data satisfy the requirement(s) of such method.

1. Unless defined, use $\alpha = 0.05$
2. Tolerance for p-value differences is 0.001
3. Skip test of homogeneity using Levene Test in lawstat package.

In [1]:
### This cell is for validation and assessment purposes. Do not modify!
### You do not need to run this cell.
library(testthat)
library(digest)
library(stringr)
library(magick)

Linking to ImageMagick 6.9.13.29
Enabled features: cairo, freetype, fftw, ghostscript, heic, lcms, pango, raw, rsvg, webp
Disabled features: fontconfig, x11



In [2]:
# Libraries for displaying the plot from file
library(png)
library(IRdisplay)

# Preparation
You will use the provided `Questionnaire2026.csv` file as the data source for the given cases. As preparation, read the file and store it as `data2026`. Make some proper adjustments if necessary. If anyone born after August 1st, 2011, consider these data as outliers and dropped them.

In [3]:
# Load the dataset into active environment
# then adjust the variables accordingly.
# Rows with NA should be removed.

data2026 <- read.csv(
    file = "Questionnaire2026.csv",
    sep = ",",
    dec = ".",
    header = TRUE,
    stringsAsFactors = FALSE
)

# Fungsi untuk merapikan format tanggal lahir yang tidak konsisten.
parse_dob <- function(x) {
    x <- trimws(as.character(x))
    x <- gsub("=", "-", x)

    dob <- as.Date(x, format = "%m/%d/%Y")

    idx <- is.na(dob)
    dob[idx] <- as.Date(x[idx], format = "%Y-%m-%d")

    idx <- is.na(dob)
    dob[idx] <- as.Date(x[idx], format = "%Y-%d-%m")

    return(dob)
}

data2026$dob <- parse_dob(data2026$dob)

# Menyesuaikan tipe data agar siap digunakan untuk analisis statistik.
data2026$batch <- as.integer(data2026$batch)
data2026$height <- as.numeric(data2026$height)
data2026$gpa <- as.numeric(data2026$gpa)
data2026$internet_usage_mb <- as.numeric(data2026$internet_usage_mb)
data2026$gender <- as.factor(data2026$gender)

# Mengambil label kelas A-F dari nama kelas.
data2026$class_group <- sub(".*\\(([A-Z])\\).*", "\\1", data2026$class)
data2026$class_group <- as.factor(data2026$class_group)

# Menghapus baris yang masih memiliki NA setelah penyesuaian data.
data2026 <- na.omit(data2026)

# Menghapus outlier tanggal lahir sesuai instruksi soal.
data2026 <- subset(data2026, dob <= as.Date("2011-08-01"))

# Mengurutkan ulang level kelas agar pasangan post-hoc rapi.
data2026$class_group <- factor(data2026$class_group, levels = sort(unique(data2026$class_group)))

str(data2026)


'data.frame':	135 obs. of  10 variables:
 $ batch            : int  2025 2025 2025 2025 2025 2025 2025 2025 2025 2025 ...
 $ class            : chr  "CSP1122 - Probability and Statistics (A)" "CSP1122 - Probability and Statistics (D)" "CSP1122 - Probability and Statistics (D)" "CSP1122 - Probability and Statistics (F)" ...
 $ gender           : Factor w/ 2 levels "Female","Male": 2 1 1 2 2 2 2 2 2 2 ...
 $ pob              : chr  "KOTA MANADO" "KOTA MANADO" "KABUPATEN MINAHASA TENGGARA" "KOTA MANADO" ...
 $ dob              : Date, format: "2008-04-24" "2007-01-10" ...
 $ height           : num  175 153 157 182 178 170 166 155 170 166 ...
 $ gpa              : num  3.7 3.65 4 3.92 3.8 3.85 3.9 3.65 3.95 0 ...
 $ research_interest: chr  "Human Informatics (Health Informatics, Multimedia, Game Development)" "Distributed Systems (Computer Networks, Security)" "Distributed Systems (Computer Networks, Security)" "Distributed Systems (Computer Networks, Security)" ...
 $ internet_usage_mb: n

In [4]:
# Cell for assessment. Do not make any attempt to modify this cell.


# Case 1: Body Height

There is a common believe that men tend to be taller than women. See if the statement is true for the first year students **(Batch 2025)** in the Undergraduate Program in Informatics UNSRAT.

Evaluation Parameters:
1. `case1.1`: `TRUE` if the male students tend to be significantly taller than the females, `FALSE` if otherwise.
2. `case1.2`: should be assigned with the p-value of the statistical test. The tolerance is 0.001.

In [5]:
# Cell for case 1
# Use the appropriate statistical test to determine whether
# the men are significantly taller than the women, or not.
#
# Do not forget to assign the evaluation parameters.

data_case1 <- subset(data2026, batch == 2025 & gender %in% c("Male", "Female"))

male_height <- data_case1$height[data_case1$gender == "Male"]
female_height <- data_case1$height[data_case1$gender == "Female"]

# Cek normalitas sebagai dasar pemilihan metode uji.
male_normality <- shapiro.test(male_height)
female_normality <- shapiro.test(female_height)

# Karena kedua kelompok memenuhi normalitas, gunakan Welch two-sample t-test.
# Alternative "greater" digunakan karena hipotesisnya laki-laki lebih tinggi.
case1_test <- t.test(
    male_height,
    female_height,
    alternative = "greater"
)

case1.1 <- case1_test$p.value < 0.05
case1.2 <- case1_test$p.value

case1.1
case1.2


[1] TRUE

[1] 1.268144e-18

In [6]:
# Cell for assessing case 1, evaluation parameter 2
# Do not modify!


In [7]:
# Cell for assessing case 1, evaluation parameter 1
# Do not modify!


# Case 2: Age Distribution (All Batches)

Check whether in all classes the students' ages are similar. Find the class(es) that has significantly different ages compared to others. The ages are calculated on May 16th, 2026.

Evaluation Parameters:
1. `case2.1`: `TRUE` if there is at least a class with a significantly different ages, `FALSE` if otherwise.
2. `case2.2`: should be assigned with the p-value of the statistical test. The tolerance is 0.001.
3. `case2.3`: a vector of the pairs of classes with significantly different ages. If none, assigned with `NULL`. Each pair should be written as a string, e.g. "AB". When there are several pairs that significantly different, then it must be ordered, according to the first class then the second one.

Note:
If a grouping criteria leads to error, it can be dropped/excluded for this particular case.

In [8]:
# Cell for case 2
# Use the appropriate statistical test to determine whether
# in all classes the students' ages are similar
#
# Do not forget to assign the evaluation parameters.

data_case2 <- data2026

reference_date <- as.Date("2026-05-16")

# Menghitung umur pada tanggal 16 Mei 2026.
data_case2$age <- as.integer(format(reference_date, "%Y")) -
    as.integer(format(data_case2$dob, "%Y")) -
    ifelse(format(reference_date, "%m-%d") < format(data_case2$dob, "%m-%d"), 1, 0)

# Cek normalitas per kelas.
age_normality <- by(data_case2$age, data_case2$class_group, shapiro.test)

# Data umur tidak normal pada beberapa kelas, sehingga digunakan Kruskal-Wallis.
case2_test <- kruskal.test(age ~ class_group, data = data_case2)

case2.1 <- case2_test$p.value < 0.05
case2.2 <- case2_test$p.value

# Post-hoc hanya perlu diisi jika hasil uji utama signifikan.
if (case2.1) {
    case2_posthoc <- pairwise.wilcox.test(
        data_case2$age,
        data_case2$class_group,
        p.adjust.method = "holm"
    )

    pair_index <- which(case2_posthoc$p.value < 0.05, arr.ind = TRUE)
    case2.3 <- c()

    for (i in seq_len(nrow(pair_index))) {
        pair <- c(
            rownames(case2_posthoc$p.value)[pair_index[i, "row"]],
            colnames(case2_posthoc$p.value)[pair_index[i, "col"]]
        )
        case2.3 <- c(case2.3, paste(sort(pair), collapse = ""))
    }

    case2.3 <- sort(case2.3)
} else {
    case2.3 <- NULL
}

case2.1
case2.2
case2.3


[1] FALSE

[1] 0.2349165

NULL

In [9]:
# Cell for assessing case 2, evaluation parameter 1
# Do not modify!


In [10]:
# Cell for assessing case 2, evaluation parameter 1
# Do not modify!


In [11]:
# Cell for assessing case 2, evaluation parameter 3
# Do not modify!


# Case 3: Height Distribution

This case is identical to Case 2, except now we use body height as the parameter. Identify whether the students' heights in all classes are similar or not.

Evaluation Parameters:
1. `case3.1`: `TRUE` if there is at least a class with a significantly different heights, `FALSE` if otherwise.
2. `case3.2`: should be assigned with the p-value of the statistical test. The tolerance is 0.001.
3. `case3.3`: a vector of the pairs of classes with significantly different heights. If none, assigned with `NULL`. Each pair should be written as a string, e.g. "AB". When there are several pairs that significantly different, then it must be ordered, according to the first class then the second one.

In [12]:
# Use the appropriate statistical test to determine whether
# in all classes the heights are similar
#
# Do not forget to assign the evaluation parameters.

data_case3 <- data2026

# Cek normalitas tinggi badan per kelas.
height_normality <- by(data_case3$height, data_case3$class_group, shapiro.test)

# Data tinggi badan per kelas memenuhi normalitas, sehingga digunakan One-way ANOVA.
case3_model <- aov(height ~ class_group, data = data_case3)
case3_anova <- summary(case3_model)

case3.1 <- case3_anova[[1]][["Pr(>F)"]][1] < 0.05
case3.2 <- case3_anova[[1]][["Pr(>F)"]][1]

# Post-hoc menggunakan Tukey HSD untuk mencari pasangan kelas yang berbeda signifikan.
if (case3.1) {
    case3_posthoc <- TukeyHSD(case3_model)$class_group
    significant_pairs <- rownames(case3_posthoc)[case3_posthoc[, "p adj"] < 0.05]

    if (length(significant_pairs) > 0) {
        case3.3 <- sapply(strsplit(significant_pairs, "-"), function(pair) {
            paste(sort(pair), collapse = "")
        })
        case3.3 <- sort(as.vector(case3.3))
    } else {
        case3.3 <- NULL
    }
} else {
    case3.3 <- NULL
}

case3.1
case3.2
case3.3


[1] TRUE

[1] 0.02755809

[1] "CD"

In [13]:
# Cell for assessing case 3, evaluation parameter 1
# Do not modify!


In [14]:
# Cell for assessing case 3, evaluation parameter 1
# Do not modify!


In [15]:
# Cell for assessing case 2, evaluation parameter 3
# Do not modify!
